# Telecom Customer Churn Predictive Analytics & Retention Strategy
### **Task 2: Machine Learning Decision Framework**

**Objective:** Build an end-to-end Machine Learning solution using regularized Cost-Sensitive Logistic Regression to estimate customer churn probability, classify accounts into *'Likely to Churn'* vs *'Likely to Stay'*, evaluate diagnostic performance, and formulate proactive retention strategies.

## 1. Environment Configuration & Data Ingestion

In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Load dataset
raw_df = pd.read_csv('Logistic Regression.csv')
print(f'Dataset successfully loaded. Dimensions: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
raw_df.head()

Dataset successfully loaded. Dimensions: 7043 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Customer Attributes Architecture & Data Hygiene
- Attribute profiling across demographics, subscription terms, and subscribed services.
- Cleaning whitespace values in `TotalCharges` using median imputation.

In [2]:
# Convert TotalCharges to numeric
raw_df['TotalCharges'] = pd.to_numeric(raw_df['TotalCharges'], errors='coerce')
missing_tc = raw_df['TotalCharges'].isnull().sum()
if missing_tc > 0:
    median_val = raw_df['TotalCharges'].median()
    raw_df['TotalCharges'] = raw_df['TotalCharges'].fillna(median_val)
    print(f'Detected {missing_tc} missing values in TotalCharges. Resolved via median imputation.')

print("\nChurn Class Proportions (%):")
print(raw_df['Churn'].value_counts(normalize=True) * 100)

Detected 11 missing values in TotalCharges. Resolved via median imputation.

Churn Class Proportions (%):
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


## 3. Key Customer Attributes Likely to Influence Churn
- **Contract Duration:** Month-to-month contracts experience significantly higher churn rates compared to One-year and Two-year contracts.
- **Lifecycle Tenure:** Early-stage customers (tenure < 12 months) represent the highest churn concentration.
- **Internet Tier:** Fiber optic customers exhibit higher defection rates than DSL users.
- **Payment Workflow:** Non-automated payment channels (Electronic check) correlate with higher attrition.
- **Value-Added Support:** Customers without TechSupport or OnlineSecurity churn at elevated rates.

In [3]:
X = raw_df.drop(columns=['customerID', 'Churn'])
y = raw_df['Churn'].map({'Yes': 1, 'No': 0})

numerical_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

feature_preprocessor = ColumnTransformer([
    ('numeric', num_pipeline, numerical_features),
    ('categorical', cat_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Partition: {X_train.shape[0]:,} samples | Testing Partition: {X_test.shape[0]:,} samples")
print("Engineered feature pipeline ready with StandardScaler and drop='first' OneHotEncoder.")

Training Partition: 5,634 samples | Testing Partition: 1,409 samples
Engineered feature pipeline ready with StandardScaler and drop='first' OneHotEncoder.


## 4. Balanced Cost-Sensitive Logistic Regression Model & Probabilities

In [4]:
churn_classifier = LogisticRegression(
    C=0.5,
    class_weight={0: 1.0, 1: 2.2},
    solver='liblinear',
    max_iter=1000,
    random_state=42
)

full_pipeline = Pipeline([
    ('preprocessing', feature_preprocessor),
    ('classifier', churn_classifier)
])

full_pipeline.fit(X_train, y_train)
print("Pipeline successfully trained with balanced cost weights {0: 1.0, 1: 2.2} and C=0.5.")

# Probability calculation
churn_probabilities = full_pipeline.predict_proba(X_test)[:, 1]
decision_threshold = 0.50
predicted_classes = (churn_probabilities >= decision_threshold).astype(int)

preview_records = pd.DataFrame({
    'Actual_Outcome': np.where(y_test.values[:10] == 1, 'Churn', 'Stay'),
    'Churn_Prob_Pct': np.round(churn_probabilities[:10] * 100, 2),
    'Assigned_Segment': np.where(predicted_classes[:10] == 1, 'Likely to Churn', 'Likely to Stay')
})
preview_records

Pipeline successfully trained with balanced cost weights {0: 1.0, 1: 2.2} and C=0.5.


,Actual_Outcome,Churn_Prob_Pct,Assigned_Segment
0,Stay,9.82,Likely to Stay
1,Stay,81.25,Likely to Churn
2,Stay,13.41,Likely to Stay
3,Stay,58.94,Likely to Churn
4,Stay,4.92,Likely to Stay
5,Stay,76.43,Likely to Churn
6,Stay,66.81,Likely to Churn
7,Stay,25.19,Likely to Stay
8,Stay,0.74,Likely to Stay
9,Churn,60.12,Likely to Churn


## 5. Predict Churn Outcome for Unseen Customer Profiles

In [5]:
synthetic_profiles = pd.DataFrame([
    {
        'Name': 'Customer Profile A (High Attrition Risk)',
        'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'No', 'Dependents': 'No',
        'tenure': 2, 'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'Fiber optic',
        'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No',
        'StreamingTV': 'Yes', 'StreamingMovies': 'No', 'Contract': 'Month-to-month',
        'PaperlessBilling': 'Yes', 'PaymentMethod': 'Electronic check',
        'MonthlyCharges': 85.50, 'TotalCharges': 171.00
    },
    {
        'Name': 'Customer Profile B (Low Attrition Risk)',
        'gender': 'Male', 'SeniorCitizen': 0, 'Partner': 'Yes', 'Dependents': 'Yes',
        'tenure': 60, 'PhoneService': 'Yes', 'MultipleLines': 'Yes', 'InternetService': 'DSL',
        'OnlineSecurity': 'Yes', 'OnlineBackup': 'Yes', 'DeviceProtection': 'Yes', 'TechSupport': 'Yes',
        'StreamingTV': 'No', 'StreamingMovies': 'No', 'Contract': 'Two year',
        'PaperlessBilling': 'No', 'PaymentMethod': 'Bank transfer (automatic)',
        'MonthlyCharges': 65.20, 'TotalCharges': 3912.00
    }
])

unseen_feat = synthetic_profiles.drop(columns=['Name'])
unseen_probs = full_pipeline.predict_proba(unseen_feat)[:, 1]
unseen_preds = (unseen_probs >= decision_threshold).astype(int)

for idx, row in synthetic_profiles.iterrows():
    action = "Deploy proactive 1-year contract discount voucher + complimentary TechSupport trial." if unseen_preds[idx] == 1 else "Routine account maintenance. Eligible for annual customer loyalty appreciation perks."
    print(f"{row['Name']}:")
    print(f"  Churn Risk: {unseen_probs[idx]*100:.2f}%")
    print(f"  Segment: {'LIKELY TO CHURN' if unseen_preds[idx] == 1 else 'LIKELY TO STAY'}")
    print(f"  Action: {action}\n")

Customer Profile A (High Attrition Risk):
  Churn Risk: 85.34%
  Segment: LIKELY TO CHURN
  Action: Deploy proactive 1-year contract discount voucher + complimentary TechSupport trial.

Customer Profile B (Low Attrition Risk):
  Churn Risk: 1.52%
  Segment: LIKELY TO STAY
  Action: Routine account maintenance. Eligible for annual customer loyalty appreciation perks.



## 6. Model Performance Evaluation

In [6]:
print('Overall Accuracy:    ', f"{accuracy_score(y_test, predicted_classes)*100:.2f}%")
print('Precision:           ', f"{precision_score(y_test, predicted_classes)*100:.2f}%")
print('Recall (Sensitivity):', f"{recall_score(y_test, predicted_classes)*100:.2f}%")
print('F1-Score:            ', f"{f1_score(y_test, predicted_classes):.4f}")
print('ROC-AUC Score:       ', f"{roc_auc_score(y_test, churn_probabilities):.4f}")
print('\nDiagnostic Report:\n', classification_report(y_test, predicted_classes, target_names=['Retained (0)', 'Churned (1)']))

Overall Accuracy:     76.15%
Precision:            53.64%
Recall (Sensitivity): 74.87%
F1-Score:             0.6250
ROC-AUC Score:        0.8421

Diagnostic Report:
               precision    recall  f1-score   support

 Retained (0)       0.89      0.77      0.82      1035
  Churned (1)       0.54      0.75      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.76      0.77      1409



## 7. Confusion Matrix Diagnostics & Misclassification Breakdown

In [7]:
cm = confusion_matrix(y_test, predicted_classes)
tn, fp, fn, tp = cm.ravel()

print(f"Correctly Identified Churn Customers (True Positives, TP): {tp}")
print(f"Non-Churn Customers Misclassified as Churn (False Positives, FP): {fp}")
print(f"Correctly Identified Retained Customers (True Negatives, TN): {tn}")
print(f"Actual Churn Customers Missed (False Negatives, FN): {fn}")

Correctly Identified Churn Customers (True Positives, TP): 280
Non-Churn Customers Misclassified as Churn (False Positives, FP): 242
Correctly Identified Retained Customers (True Negatives, TN): 793
Actual Churn Customers Missed (False Negatives, FN): 94


## 8. Business Interpretation
### **Is it better to wrongly flag a loyal customer as churn or miss an actual churner?**

- **Wrongly flagging a loyal customer (False Positive):**
  - *Cost:* Unnecessary marketing outreach effort or promotional incentive.
  - *Impact:* Minimal downside; loyal customers often feel appreciated.
- **Missing a customer who is actually going to leave (False Negative):**
  - *Cost:* Permanent loss of recurring subscription revenue plus additional customer acquisition costs to replace the departed customer.
  - *Impact:* Direct erosion of recurring revenue and market share.
- **Conclusion:** Missing an actual churn customer (False Negative) is **far more detrimental**. A False Positive may result in unnecessary retention costs, while a False Negative can result in lost revenue and additional customer acquisition costs. By tuning the cost weights, our model captures **~75% of churners (280 accounts)** while keeping false alarms lower than a naive balanced model.

## 9. Suggested Improvements for Model Reliability
1. **Cost-Curve Threshold Optimization:** Tune the classification threshold dynamically based on the exact dollar ratio between retention offer costs and replacement customer acquisition costs.
2. **Non-Linear Tree Models:** Benchmark Gradient Boosted Trees (XGBoost, LightGBM) to capture non-linear service bundle interactions.
3. **Behavioral Telemetry Integration:** Incorporate customer service call frequencies, broadband speed drop complaints, and recent billing dispute histories.